# CS-681 Project: Integrated Demo (Gradio)
Fundus image -> MedGemma report (two prompt strategies) -> spoken summary (SpeechT5).

**Run this notebook interactively (Run All), not with Save Version.** The app runs only while the session is on.

## 1. Install

In [1]:
!pip install -q -U transformers accelerate sentencepiece soundfile gradio "datasets<4.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 92.7 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 51.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.5/64.5 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.7/146.7 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 30.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.48.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not 

## 2. Hugging Face login (MedGemma is gated)

In [2]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

login(UserSecretsClient().get_secret("HF_TOKEN"))

## 3. Load MedGemma

In [3]:
import re
import numpy as np
import torch
from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText

torch.manual_seed(42)
MODEL_ID = "google/medgemma-4b-it"
model = AutoModelForImageTextToText.from_pretrained(MODEL_ID, dtype=torch.bfloat16, device_map="auto")
processor = AutoProcessor.from_pretrained(MODEL_ID)
print("MedGemma loaded on", model.device)

config.json:   0%|          | 0.00/2.47k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/90.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 33.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

MedGemma loaded on cuda:0


## 4. The same two prompts used in the experiments

In [4]:
PROMPT_ZERO_SHOT = ("Describe the findings in this retinal fundus image and "
                    "assess the diabetic retinopathy severity.")

PROMPT_STRUCTURED = """Analyze this retinal fundus image and respond ONLY in the exact format below.

Use the ICDR diabetic retinopathy scale:
0 = No DR: no abnormalities
1 = Mild NPDR: microaneurysms only
2 = Moderate NPDR: more than microaneurysms but less than severe
3 = Severe NPDR: extensive hemorrhages, venous beading, or IRMA, with no signs of proliferation
4 = Proliferative DR: neovascularization and/or vitreous/preretinal hemorrhage

IMAGE QUALITY: <Good / Adequate / Poor>
FINDINGS:
- Microaneurysms: <Present / Absent>
- Hemorrhages: <Present / Absent>
- Hard exudates: <Present / Absent>
- Cotton wool spots: <Present / Absent>
- Venous beading or IRMA: <Present / Absent>
- Neovascularization: <Present / Absent>
- Vitreous or preretinal hemorrhage: <Present / Absent>
SEVERITY GRADE: <a single number from 0 to 4>
SEVERITY LABEL: <No DR / Mild NPDR / Moderate NPDR / Severe NPDR / Proliferative DR>
RECOMMENDATION: <one actionable follow-up sentence>
SUMMARY: <2-3 simple sentences for a spoken audio summary>"""

def ask_medgemma(prompt, image=None, max_new_tokens=400):
    """Run MedGemma with an optional image and return the generated text."""
    content = [{"type": "text", "text": prompt}]
    if image is not None:
        content.append({"type": "image", "image": image})
    messages = [
        {"role": "system", "content": [{"type": "text", "text": "You are an expert ophthalmologist."}]},
        {"role": "user", "content": content}]
    inputs = processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=True,
                                           return_dict=True, return_tensors="pt").to(model.device, dtype=torch.bfloat16)
    n = inputs["input_ids"].shape[-1]
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return processor.decode(out[0][n:], skip_special_tokens=True).strip()

## 5. Load SpeechT5 and the text normalization

In [5]:
from transformers import SpeechT5Processor, SpeechT5ForTextToSpeech, SpeechT5HifiGan
from datasets import load_dataset

tts_proc = SpeechT5Processor.from_pretrained("microsoft/speecht5_tts")
tts = SpeechT5ForTextToSpeech.from_pretrained("microsoft/speecht5_tts").to("cuda").eval()
vocoder = SpeechT5HifiGan.from_pretrained("microsoft/speecht5_hifigan").to("cuda").eval()
xv = load_dataset("Matthijs/cmu-arctic-xvectors", split="validation", trust_remote_code=True)
speaker = torch.tensor(xv[7306]["xvector"]).unsqueeze(0).to("cuda")
SR = 16000

ABBREV = {r"\bNPDR\b": "non proliferative diabetic retinopathy",
          r"\bPDR\b": "proliferative diabetic retinopathy",
          r"\bDR\b": "diabetic retinopathy",
          r"\bIRMA\b": "intraretinal microvascular abnormalities"}

def normalize_for_tts(text):
    """Expand abbreviations so SpeechT5 can pronounce them."""
    for p, full in ABBREV.items():
        text = re.sub(p, full, text)
    return re.sub(r"\s+", " ", text.replace("non-proliferative", "non proliferative")).strip()

def speak(text):
    """Sentence-by-sentence SpeechT5 synthesis joined with short pauses."""
    pause = np.zeros(int(SR * 0.25), dtype=np.float32)
    parts = []
    for s in [s for s in re.split(r"(?<=[.!?])\s+", text) if s.strip()]:
        ids = tts_proc(text=s, return_tensors="pt")["input_ids"].to("cuda")
        with torch.inference_mode():
            wav = tts.generate_speech(ids, speaker, vocoder=vocoder)
        parts += [wav.cpu().numpy().astype(np.float32), pause]
    return np.concatenate(parts) if parts else np.zeros(SR, dtype=np.float32)
print("TTS ready")

preprocessor_config.json:   0%|          | 0.00/433 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.06k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/232 [00:00<?, ?B/s]

spm_char.model: reconstructing file:   0%|          |  0.00B /  238kB            

spm_char.model: downloading bytes:           |  0.00B            

...

cmu-arctic-xvectors.py:   0%|          | 0.00/1.36k [00:00<?, ?B/s]

default/validation/0000.parquet: reconstructing file:   0%|          |  0.00B / 21.3MB            

default/validation/0000.parquet: downloading bytes:           |  0.00B            

Generating validation split:   0%|          | 0/7931 [00:00<?, ? examples/s]

TTS ready


## 6. Pipeline function used by the app

In [6]:
LABELS = {0: "No DR", 1: "Mild NPDR", 2: "Moderate NPDR", 3: "Severe NPDR", 4: "Proliferative DR"}

def analyze(image, strategy):
    """Image + prompt strategy -> (grade badge, full report, (sample_rate, audio))."""
    if image is None:
        return "Please upload a fundus image.", "", None
    image = image.convert("RGB")
    if strategy == "Structured template":
        report = ask_medgemma(PROMPT_STRUCTURED, image, max_new_tokens=350)
        m = re.search(r"SUMMARY:\s*(.+)", report, re.S)
        summary = m.group(1).strip() if m else report[:400]
        g = re.search(r"SEVERITY GRADE:\s*([0-4])", report)
        badge = f"### Predicted grade: {g.group(1)} ({LABELS[int(g.group(1))]})" if g else "### Grade not found"
    else:
        report = ask_medgemma(PROMPT_ZERO_SHOT, image, max_new_tokens=512)
        # Zero-shot has no summary section, so ask MedGemma (text only) for a short spoken summary
        summary = ask_medgemma("Summarize this retinal report in 2-3 simple sentences for a spoken summary. "
                               "Return only the summary.\n\n" + report, max_new_tokens=120)
        badge = "### Zero-shot: free-text report (no fixed grade field)"
    audio = speak(normalize_for_tts(summary))
    return badge, report, (SR, audio)

## 7. Launch the app

In [7]:
import glob
import gradio as gr

# A few example images from the attached dataset (one per grade if available)
examples = []
try:
    import pandas as pd
    s = pd.read_csv(glob.glob("/kaggle/input/**/sample_250.csv", recursive=True)[0])
    examples = [[p, "Structured template"] for p in s.groupby("level").head(1)["path"]]
except Exception as e:
    print("No examples:", e)

DISCLAIMER = ("**Disclaimer:** research prototype for a course project. It is **not** a medical device "
              "and must not be used for diagnosis. Always consult a qualified ophthalmologist.")

with gr.Blocks(title="DR Report & Audio Generator") as demo:
    gr.Markdown("# Diabetic Retinopathy Report & Audio Generator\n"
                "Upload a retinal fundus image to get a MedGemma report and a spoken summary (SpeechT5).")
    gr.Markdown(DISCLAIMER)
    with gr.Row():
        with gr.Column():
            img = gr.Image(type="pil", label="Fundus image")
            strat = gr.Radio(["Structured template", "Zero-shot"], value="Structured template",
                             label="Prompting strategy")
            btn = gr.Button("Analyze", variant="primary")
        with gr.Column():
            badge = gr.Markdown()
            report = gr.Textbox(label="Generated report", lines=16)
            audio = gr.Audio(label="Spoken summary", type="numpy")
    btn.click(analyze, inputs=[img, strat], outputs=[badge, report, audio])
    if examples:
        gr.Examples(examples=examples, inputs=[img, strat])

# share=True gives a public link that works while this session is running
demo.launch(share=True, debug=False)

No examples: list index out of range
* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://81a0bf6f69df50609d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
